# **TensorFlow**

## **TensorFlow v1 (Legacy Version)**

In [3]:
# Example of running TensorFlow v1 code in TensorFlow v2
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()  # Disable TensorFlow v2 behavior

# Define the computation graph using TensorFlow v1 style
x = tf.placeholder(tf.float32)
y = tf.placeholder(tf.float32)
result = x + y

# Create a session and execute the graph
with tf.Session() as sess:
    print(sess.run(result, feed_dict={x: 1, y: 2}))

Instructions for updating:
non-resource variables are not supported in the long term


3.0


## **TensorFlow v2 (Modern Version)**

In [2]:
# Example of TensorFlow v2
import tensorflow as tf

# TensorFlow v2 uses eager execution by default
x = tf.constant(1)
y = tf.constant(2)
result = x + y
print(result)  # Directly outputs the result (no need for a session)

tf.Tensor(3, shape=(), dtype=int32)


Support for Keras

In [4]:
# Example of defining a model with Keras in TensorFlow v2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation='relu', input_shape=(100,)),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Backward Compatibility

In [5]:
# Running TensorFlow v1 code in TensorFlow v2
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()  # Disable TensorFlow v2 behavior

# Now, TensorFlow v1 code can be used as normal

# **Introduction to Transfer Learning**

Transfer Learning in Keras (Using Pre-trained VGG16)  "path_to_train_directory" TASK

In [8]:
# from tensorflow.keras.applications import VGG16
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Flatten
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# # Load the pre-trained VGG16 model, excluding the top fully connected layers
# base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# # Freeze the base model's layers
# for layer in base_model.layers:
#     layer.trainable = False

# # Create a new model on top of the frozen layers
# model = Sequential([
#     base_model,
#     Flatten(),  # Flatten the output of the last convolutional block
#     Dense(128, activation='relu'),
#     Dense(10, activation='softmax')  # Adjust the number of classes as needed
# ])

# # Compile the model
# model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# # Data Preparation (Assuming you have image datasets in 'train' and 'validation' directories)
# train_datagen = ImageDataGenerator(rescale=1./255)
# val_datagen = ImageDataGenerator(rescale=1./255)

# train_generator = train_datagen.flow_from_directory(
#     'path_to_train_directory',  # Replace with your training data directory
#     target_size=(224, 224),
#     batch_size=32,
#     class_mode='categorical'
# )

# validation_generator = val_datagen.flow_from_directory(
#     'path_to_validation_directory',  # Replace with your validation data directory
#     target_size=(224, 224),
#     batch_size=32,
#     class_mode='categorical'
# )

# # Train the model
# history = model.fit(train_generator, validation_data=validation_generator, epochs=10)

# # Output the result after training
# print(f"Training completed after {len(history.epoch)} epochs.")

# **Hyperparameter Tuning**

**Random Search with Keras Tuner**

In [1]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)

In [2]:
# Install Keras Tuner if not already installed
!pip install keras-tuner

# Import necessary libraries
from kerastuner.tuners import RandomSearch
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

# Ensure eager execution is enabled (default in TensorFlow 2.x)
tf.config.run_functions_eagerly(True)

# Load the MNIST dataset
(X_train, y_train), (X_val, y_val) = mnist.load_data()

# Normalize the data (scale it between 0 and 1)
X_train = X_train.astype('float32') / 255
X_val = X_val.astype('float32') / 255

# One-hot encode the labels (because it's a classification problem)
y_train = to_categorical(y_train, 10)
y_val = to_categorical(y_val, 10)

# Define the model for Keras Tuner
def build_model(hp):
    model = Sequential()
    model.add(Flatten(input_shape=(28, 28)))  # Flatten the input images

    # Tune the number of units in the Dense layer
    model.add(Dense(units=hp.Int('units', min_value=32, max_value=256, step=32), activation='relu'))

    # Output layer for 10 classes
    model.add(Dense(10, activation='softmax'))

    # Tune the learning rate for Adam optimizer
    model.compile(optimizer=Adam(learning_rate=hp.Choice('learning_rate', [0.001, 0.01, 0.0001])),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Initialize the Keras Tuner for Random Search
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,  # Fewer trials to keep it simple
    executions_per_trial=1,  # Number of executions per trial
    directory='simple_tuning',  # Directory to save the results
    project_name='mnist_simple'
)

# Start the search for the best hyperparameters
# Include shuffle=True and specify a batch_size to ensure proper batching of data
tuner.search(X_train, y_train, epochs=3, validation_data=(X_val, y_val), shuffle=True, batch_size=32)

# Get the best model found during the tuning
best_model = tuner.get_best_models(num_models=1)[0]

# Print the summary of the best model
best_model.summary()


Trial 3 Complete [00h 04m 01s]
val_accuracy: 0.9753999710083008

Best val_accuracy So Far: 0.9753999710083008
Total elapsed time: 00h 13m 02s


/usr/local/lib/python3.10/dist-packages/keras/src/saving/saving_lib.py:576: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten (Flatten)                    │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 160)                 │         125,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,610 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 127,210 (496.91 KB)

 Trainable params: 127,210 (496.91 KB)

 Non-trainable params: 0 (0.00 B)

# **Early Stopping and Model Checkpointing**

In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load the MNIST dataset
(X_train, y_train), (X_val, y_val) = mnist.load_data()

# Normalize the data (scale it between 0 and 1)
X_train = X_train.astype('float32') / 255
X_val = X_val.astype('float32') / 255

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_val = to_categorical(y_val, 10)

# Define a simple neural network model
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')  # 10 output classes
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Early stopping to stop training when validation loss stops improving
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Model checkpoint to save the best model during training
model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

# Train the model with early stopping and model checkpointing
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, callbacks=[early_stopping, model_checkpoint])

# The model training is complete, and the best model is saved

Epoch 1/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 41ms/step - accuracy: 0.8800 - loss: 0.4306 - val_accuracy: 0.9603 - val_loss: 0.1337
Epoch 2/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 42ms/step - accuracy: 0.9642 - loss: 0.1221 - val_accuracy: 0.9695 - val_loss: 0.1017
Epoch 3/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 41ms/step - accuracy: 0.9756 - loss: 0.0777 - val_accuracy: 0.9742 - val_loss: 0.0818
Epoch 4/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.9839 - loss: 0.0543 - val_accuracy: 0.9758 - val_loss: 0.0789
Epoch 5/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 42ms/step - accuracy: 0.9872 - loss: 0.0420 - val_accuracy: 0.9748 - val_loss: 0.0859
Epoch 6/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 41ms/step - accuracy: 0.9896 - loss: 0.0336 - val_accuracy: 0.9791 - val_loss: 0.0711
Epoch 7/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 41ms/step - accuracy: 0.9919 - loss: 0.0257 - val_accuracy: 0.9766 - val_loss: 0.0799
Epoch 8/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 42ms/step - accuracy: 0.9940 -